# 02 Model Convert

Convert `best_line_follower_model_xy.onnx` to Horizon `.bin` in `/root/originbot`.

In [ ]:
import os
import shutil
from pathlib import Path

project_dir = Path.cwd().resolve()
mapper_dir = Path(os.environ.get("MAPPER_DIR", "/root/originbot")).resolve()

required_items = [
    "best_line_follower_model_xy.onnx",
    "image_dataset",
    "prepare_horizon_mapper.py",
    "run_horizon_convert.sh",
    "02_preprocess.sh",
    "03_build.sh",
    "resnet18_224x224_nv12.yaml",
]

if not mapper_dir.is_dir():
    raise FileNotFoundError(
        f"Mapper directory does not exist: {mapper_dir}\n"
        "Set env MAPPER_DIR to your workspace path."
    )

missing = [name for name in required_items if not (mapper_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files in mapper directory:\n"
        + "\n".join(f" - {name}" for name in missing)
    )

hb_mapper_path = shutil.which("hb_mapper")
if hb_mapper_path is None:
    raise RuntimeError(
        "hb_mapper is not in PATH. Please enter Horizon OpenExplorer environment/container first."
    )

print("project_dir:", project_dir)
print("mapper_dir:", mapper_dir)
print("hb_mapper:", hb_mapper_path)
print("Preflight check passed.")

In [ ]:
import subprocess

prepare_cmd = [
    "python", str(mapper_dir / "prepare_horizon_mapper.py"),
    "--mapper-dir", str(mapper_dir),
    "--onnx", str(mapper_dir / "best_line_follower_model_xy.onnx"),
    "--dataset-dir", str(mapper_dir / "image_dataset"),
]
print("RUN:", " ".join(prepare_cmd))
subprocess.run(prepare_cmd, check=True)

In [ ]:
import subprocess

convert_cmd = ["bash", str(mapper_dir / "run_horizon_convert.sh"), str(mapper_dir)]
print("RUN:", " ".join(convert_cmd))
subprocess.run(convert_cmd, check=True)

In [ ]:
import glob

bin_files = glob.glob(str(mapper_dir / "model_output" / "*.bin"))
print("bin_files:")
for file_path in bin_files:
    print(" -", file_path)
if not bin_files:
    raise RuntimeError(
        f"No .bin generated under {mapper_dir / 'model_output'}.\n"
        "Check previous cell output, especially hb_mapper logs."
    )